# Concrete Strength Prediction — Training on Colab

**Workflow:**
1. Mount Google Drive for persistent storage
2. Clone/pull repo from GitHub
3. Install package
4. Run experiments
5. Results auto-saved to Drive

## 1. Setup (run once per session)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive (persistent between sessions)
!mkdir -p /content/drive/MyDrive/concrete_project/checkpoints
!mkdir -p /content/drive/MyDrive/concrete_project/experiments
!mkdir -p /content/drive/MyDrive/concrete_project/neat_cache

print('Drive mounted. Persistent storage ready.')

In [ ]:
# Clone or pull the repo
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/concrete-strength.git'  # <-- CHANGE THIS
REPO_DIR = '/content/concrete-strength'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install package
!pip install -e '.[colab]' -q

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Symlink Drive directories for persistent storage
!ln -sf /content/drive/MyDrive/concrete_project/checkpoints experiments/checkpoints_drive
!ln -sf /content/drive/MyDrive/concrete_project/experiments experiments/log_drive

# Quick smoke test
!python run_experiment.py --mode smoke_test

## 2. Experiments

### 2a. Supervised Grid Search (~30-50 min on T4)

In [ ]:
# Supervised hyperparameter search (81 configs)
# Output dir on Drive for persistence
!python run_experiment.py \
    --mode supervised_grid \
    --output_dir /content/drive/MyDrive/concrete_project/experiments

### 2b. GAN Tuning (~75 min on T4)

In [ ]:
# GAN tuning on top-3 supervised configs
!python run_experiment.py \
    --mode gan_tune \
    --top_k 3 \
    --output_dir /content/drive/MyDrive/concrete_project/experiments

### 2c. Full Pipeline (best config, long training)

In [ ]:
# Full training with best config
!python run_experiment.py \
    --mode full_pipeline \
    --epochs 1000 \
    --output_dir /content/drive/MyDrive/concrete_project/experiments

## 3. Results

In [ ]:
# View experiment log
from materialgen.tracker import ExperimentTracker
tracker = ExperimentTracker('/content/drive/MyDrive/concrete_project/experiments')
print(tracker.summary_table())

# Best run
best = tracker.best_run(metric='mae')
if best:
    print(f'\nBest: {best["experiment"]}')
    print(f'  MAE={best["metrics"]["mae"]:.2f}, R2={best["metrics"]["r2"]:.4f}')
    print(f'  Config: {best["config"]}')

In [ ]:
# Copy best model to Drive for safekeeping
import shutil
import glob

best_models = glob.glob('experiments/checkpoints/*.pt')
for m in best_models[:5]:  # top 5
    dst = f'/content/drive/MyDrive/concrete_project/checkpoints/{os.path.basename(m)}'
    shutil.copy2(m, dst)
    print(f'Saved: {dst}')

## 4. Commit results back to Git

In [ ]:
# Commit experiment results (JSON only, not model weights)
!git add experiments/*.jsonl experiments/checkpoints/*.json
!git commit -m 'Add experiment results from Colab'
!git push